# Predicting Online Purchases — data, EDA and baselines

This notebook implements the first approved modelling stage for the UCI Online Shoppers Purchasing Intention data. It stops after the four all-predictor diagnostics and four operational baselines. No revised specifications, model selection, threshold analysis or held-out evaluation are included in this baseline stage.

**Research question.** How well can session variables predict an observed purchase, and which of the course classification approaches merits further comparison? The outcome is historical session `Revenue`; results do not estimate voucher effects.

## 0. Fixed protocol and purpose

**Purpose.** Establish the evaluation rules before any model results. **Method.** Retain all records, split once into stratified 80% training and 20% holdout with seed 42, then use identical shuffled five-fold stratified training folds. All EDA and baseline comparisons in this notebook use training rows only. **Implication.** Accuracy alone is misleading at a 15.5% purchase rate; compare ranking with ROC-AUC and report average precision as support. The held-out partition is reserved for a later frozen-winner audit. A prior 75/25 analysis examined this same dataset, so the new holdout is not historically untouched.

In [ ]:
from pathlib import Path
import hashlib, json, warnings, platform, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy, sklearn
from scipy import stats
from IPython.display import display, Markdown
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.exceptions import ConvergenceWarning
import statsmodels.api as sm
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss

RUN_START=time.perf_counter()
SEED=42; TEST_FRAC=.20; K=5
TARGET='Revenue'
NUMERIC_ALL=['Administrative','Administrative_Duration','Informational','Informational_Duration',
 'ProductRelated','ProductRelated_Duration','BounceRates','ExitRates','PageValues','SpecialDay']
CAT=['Month','OperatingSystems','Browser','Region','TrafficType','VisitorType']
ACTIVITY=['Administrative','Administrative_Duration','Informational','Informational_Duration','ProductRelated','ProductRelated_Duration']
NUMERIC_SAFE=[x for x in NUMERIC_ALL if x!='PageValues']
ALL=NUMERIC_ALL+CAT+['Weekend']; SAFE=NUMERIC_SAFE+CAT+['Weekend']
EXPECTED=set(ALL+[TARGET])
root_candidates=[Path('data/uci_468.csv'),Path('../data/uci_468.csv')]
data_path=next((p for p in root_candidates if p.is_file()),None)
if data_path is None: raise FileNotFoundError('Run from repository root or a direct child directory; expected data/uci_468.csv')
ROOT=data_path.parent.parent if data_path.parent.name=='data' else data_path.parent.parent
ARTIFACTS=ROOT/'artifacts'; ARTIFACTS.mkdir(exist_ok=True)
WORK=ROOT/'work'; WORK.mkdir(exist_ok=True)
PROGRESS_LOG=WORK/'baseline_progress.jsonl'; PROGRESS_LOG.write_text('')
def mark_progress(candidate,fold,event,**details):
    record={'candidate':candidate,'fold':fold,'event':event,'elapsed_seconds':round(time.perf_counter()-RUN_START,3),**details}
    with PROGRESS_LOG.open('a') as f: f.write(json.dumps(record,default=str)+'\n')
print(f'Python {platform.python_version()} | pandas {pd.__version__} | NumPy {np.__version__} | SciPy {scipy.__version__} | scikit-learn {sklearn.__version__}')
print('Primary selection metric is mean five-fold training ROC-AUC; AP and fold variation are supporting evidence. No selection occurs in this notebook.')

## 1. Data quality, split and row provenance (A1)

**Purpose.** Verify that the source is complete and internally consistent, and establish exact reproducible row membership before examining predictors. **Method.** Check shape, columns, types, numeric finiteness, missing and blank nominal values, duplicates and class balance; retain all sessions. Split row IDs once, store them, sort training rows by original file order, and materialise shared five-fold row-ID mappings. **Implication.** Identical records are not evidence of identical visitors; duplicates remain. All predictor summaries below use the 80% training partition.

In [ ]:
raw=pd.read_csv(data_path); raw.insert(0,'_row_id',np.arange(len(raw),dtype=int))
source_hash=hashlib.sha256(data_path.read_bytes()).hexdigest()
if raw.shape!=(12330,19): raise ValueError(f'Expected 12,330 rows and 18 source columns, found {raw.shape}')
if set(raw.columns)-{'_row_id'} != EXPECTED: raise ValueError(f'Unexpected source columns: {sorted((set(raw.columns)-{"_row_id"}) ^ EXPECTED)}')
if raw[TARGET].isna().any() or not set(raw[TARGET].unique()).issubset({0,1,False,True}): raise ValueError('Revenue must be complete binary 0/1')
raw[TARGET]=raw[TARGET].astype(int)
if raw[NUMERIC_ALL].isna().any().any() or not np.isfinite(raw[NUMERIC_ALL].to_numpy(float)).all(): raise ValueError('Numeric source predictors must be complete and finite')
blank=raw[CAT].astype('string').apply(lambda s:s.str.strip().eq('').sum())
missing=raw.drop(columns='_row_id').isna().sum()
if missing.any() or blank.any(): raise ValueError('Unexpected missing values or blank nominal markers; inspect source before modelling')
if raw['Weekend'].isna().any() or not set(raw['Weekend'].unique()).issubset({0,1,False,True}): raise ValueError('Weekend must be complete binary 0/1')
quality=pd.DataFrame({'dtype':raw.drop(columns='_row_id').dtypes.astype(str),'missing':missing,
                      'blank nominal':blank.reindex(missing.index,fill_value=0),'unique':raw.drop(columns='_row_id').nunique()})
print('Source shape (rows, columns):',(len(raw),len(raw.columns)-1),'SHA-256:',source_hash)
print('Duplicate excess / unique full rows:',int(raw.duplicated(subset=raw.columns.drop('_row_id')).sum()),'/',raw.drop(columns='_row_id').drop_duplicates().shape[0])
display(quality)
display(raw[TARGET].value_counts().sort_index().rename_axis(TARGET).to_frame('sessions').assign(share=lambda d:d.sessions/len(raw)))
dup=raw.loc[raw.duplicated(subset=raw.columns.drop('_row_id'),keep='first')]
print(f'Excess duplicates: {len(dup)}; duplicate outcomes:',dup[TARGET].value_counts().sort_index().to_dict())
display(pd.DataFrame({'duplicate median activity':dup[ACTIVITY].median(),'duplicate zero share':dup[ACTIVITY].eq(0).mean()}))
print('Reverified duplicate description: duplicate rows are nonbuyers with low activity summaries; all rows remain. This does not identify unique people or repeated sessions.')

train_ids,test_ids=train_test_split(raw['_row_id'].to_numpy(),test_size=TEST_FRAC,
    stratify=raw[TARGET].to_numpy(),random_state=SEED)
train_ids=np.sort(train_ids); test_ids=np.sort(test_ids)
pd.DataFrame({'row_id':train_ids,'partition':'train'}).to_csv(ARTIFACTS/'train_row_ids.csv',index=False)
pd.DataFrame({'row_id':test_ids,'partition':'test'}).to_csv(ARTIFACTS/'test_row_ids.csv',index=False)
train=raw.set_index('_row_id').loc[train_ids].copy()
y_train=train.pop(TARGET).astype(int)
X_train=train[ALL].copy()
X_train[CAT]=X_train[CAT].astype('string')
X_train['Weekend']=X_train['Weekend'].astype(int)
del raw,dup
cv=StratifiedKFold(n_splits=K,shuffle=True,random_state=SEED)
FOLD_INDICES=list(cv.split(np.zeros(len(y_train)),y_train))
fold_map=[]
for fold,(fit_idx,val_idx) in enumerate(FOLD_INDICES,1):
    fold_map += [{'row_id':int(X_train.index[i]),'fold':fold,'role':'fit'} for i in fit_idx]
    fold_map += [{'row_id':int(X_train.index[i]),'fold':fold,'role':'validation'} for i in val_idx]
pd.DataFrame(fold_map).to_csv(ARTIFACTS/'cv_fold_row_ids.csv',index=False)
print(f'Training={len(X_train):,}; holdout IDs recorded={len(test_ids):,}; purchases={y_train.sum():,} ({y_train.mean():.2%}).')
print('Shared validation fold sizes:',[len(v) for _,v in FOLD_INDICES],'; row-ID order is sorted original CSV order.')
print('The holdout is reserved: no predictor/outcome summaries, fitted transforms, or evaluation are performed on those rows.')

## 2. Numerical training EDA (A2)

**Purpose.** Identify skewed, zero-heavy session measures that may challenge Gaussian likelihoods or suggest alternative representations. **Method.** Summarise training medians, zero shares and skewness; plot selected raw distributions with log-scaled counts. **Implication.** EDA generates fixed hypotheses for later recipe review; it does not select transforms by model performance.

In [ ]:
Xtr=X_train; ytr=y_train
num_summary=pd.DataFrame({'median':Xtr[NUMERIC_ALL].median(),'zero share':Xtr[NUMERIC_ALL].eq(0).mean(),
                          'skew':Xtr[NUMERIC_ALL].skew(),'buyer median':Xtr.loc[ytr.eq(1),NUMERIC_ALL].median(),
                          'nonbuyer median':Xtr.loc[ytr.eq(0),NUMERIC_ALL].median()})
display(num_summary.round(3))
fig,axes=plt.subplots(1,2,figsize=(8,3.5))
for ax,col in zip(axes.flat,['Informational_Duration','ProductRelated_Duration']):
    ax.hist(Xtr[col],bins=35,log=True,color='#4779a8'); ax.set(title=col,xlabel='Recorded value',ylabel='Training sessions (log count)')
plt.tight_layout(); plt.show()
long_tail=num_summary.loc[ACTIVITY,'skew']
print('Five of six activity fields have skew above 2; Administrative count is just below at {:.2f}.'.format(long_tail['Administrative']))
print('Informational count/duration are zero in {:.1%}/{:.1%} of training sessions, with buyer rates {:.1%}/{:.1%} at zero versus {:.1%}/{:.1%} when positive.'.format(
 num_summary.loc['Informational','zero share'],num_summary.loc['Informational_Duration','zero share'],
 ytr[Xtr.Informational.eq(0)].mean(),ytr[Xtr.Informational_Duration.eq(0)].mean(),
 ytr[Xtr.Informational.gt(0)].mean(),ytr[Xtr.Informational_Duration.gt(0)].mean()))
print('Implication: the long tails motivate reviewing the preapproved activity log1p form; only informational count and duration meet the training ≥50% zero-share rule for fold-fitted presence indicators. No model comparison is made here.')

## 3. Categorical training EDA (A3)

**Purpose.** Check whether purchase rates vary across nominal groups and whether sparse levels make those rates unstable. **Method.** Display level counts and buyer rates for each nominal feature and selected VisitorType×Month support. **Implication.** Treat nominal values as unranked categories; sparse rates do not justify arbitrary pooling or interaction terms.

In [ ]:
cat_summary=[]
for col in CAT:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))
    cat_summary.append({'feature':col,'levels':len(tab),'small levels <50':int((tab.sessions<50).sum()),
      'small-level sessions':int(tab.loc[tab.sessions<50,'sessions'].sum()),
      'buyer-rate range':f"{tab.buyer_rate.min():.1%}–{tab.buyer_rate.max():.1%}"})
display(pd.DataFrame(cat_summary).set_index('feature'))
for col in ['Month','VisitorType']:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean')).sort_values('sessions',ascending=False)
    print(f'Training group counts and observed buyer rates: {col}')
    display(tab.round(3))
rare=[]
for col in ['TrafficType','Browser','Region']:
    tab=Xtr.assign(Revenue=ytr).groupby(col,observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))
    small=tab[tab.sessions<50]
    rare.append({'feature':col,'levels below n=50':len(small),'total levels':len(tab),
                 'rare-cell rate range':f"{small.buyer_rate.min():.1%}–{small.buyer_rate.max():.1%}" if len(small) else 'none'})
print('Rare-category evidence (summary only; detailed training-only investigation is in supplementary/training_eda.ipynb):')
display(pd.DataFrame(rare).set_index('feature'))
vtm=Xtr.assign(Revenue=ytr).groupby(['VisitorType','Month'],observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean')).reset_index()
print(f"VisitorType × Month has {int((vtm.sessions<50).sum())} cells below n=50 out of {len(vtm)} observed; the cross-cell display/rates are retained in the supplementary EDA only.")
special=Xtr.assign(Revenue=ytr).groupby('SpecialDay',observed=True).agg(sessions=('Revenue','size'),buyers=('Revenue','sum'),buyer_rate=('Revenue','mean'))
display(special.round(4))
print('SpecialDay zero/positive purchase rates: {:.1%}/{:.1%}; positive score rates are nonmonotone. Keep numeric baseline; presence is considered only for reviewed NB revision.'.format(ytr[Xtr.SpecialDay.eq(0)].mean(),ytr[Xtr.SpecialDay.gt(0)].mean()))
print('Implication: nominal predictors require categorical encoding; sparse level/cross-cell rates are uncertain and do not motivate arbitrary pooling or an interaction.')

## 4. Relationships and assumption checks (A4)

**Purpose.** Identify dependence and shape features that strain model assumptions without treating marginal diagnostics as formal tests. **Method.** Compare selected numeric correlations within each class, inspect representative class-specific tails and tie-safe marginal empirical logits. **Implication.** Dependence challenges Naive Bayes conditional independence. LDA permits correlated predictors; differing selected correlations can suggest covariance heterogeneity, while long tails and mixed inputs challenge its Gaussian ideal. None of these summaries proves unequal covariance or conditional logit nonlinearity.

In [ ]:
NUM_SAFE=[c for c in NUMERIC_ALL if c!='PageValues']+['Weekend']
for cls in [0,1]:
    corr=Xtr.loc[ytr.eq(cls),NUM_SAFE].corr().abs()
    pairs=corr.where(np.triu(np.ones(corr.shape),k=1).astype(bool)).stack().sort_values(ascending=False).head(5)
    display(pairs.rename(f'class {cls}: top absolute numeric correlations').to_frame())
print('BounceRates/ExitRates correlations are {:.3f} for nonbuyers and {:.3f} for buyers; ProductRelated/ProductRelated_Duration are {:.3f} and {:.3f}. These selected differences do not establish unequal full covariance matrices.'.format(
 Xtr.loc[ytr.eq(0),['BounceRates','ExitRates']].corr().iloc[0,1],Xtr.loc[ytr.eq(1),['BounceRates','ExitRates']].corr().iloc[0,1],
 Xtr.loc[ytr.eq(0),['ProductRelated','ProductRelated_Duration']].corr().iloc[0,1],Xtr.loc[ytr.eq(1),['ProductRelated','ProductRelated_Duration']].corr().iloc[0,1]))
fig,axes=plt.subplots(1,2,figsize=(8,3.5))
for ax,col in zip(axes,['ProductRelated_Duration','Informational_Duration']):
    for cls,label,color in [(0,'No purchase','#4779a8'),(1,'Purchase','#d97941')]:
        vals=Xtr.loc[ytr.eq(cls),col].to_numpy()
        (osm,osr),_=stats.probplot(vals,dist='norm',fit=True)
        ax.plot(osm,osr,'.',ms=2,alpha=.35,label=label,color=color)
    ax.set(title=f'{col}: normal Q–Q by class',xlabel='Normal theoretical quantile',ylabel='Observed quantile'); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

def marginal_elogit(col, bins=8):
    z=pd.DataFrame({'x':Xtr[col],'buyer':ytr}); z['bin']='zero'; positive=z.x.gt(0)
    if positive.any(): z.loc[positive,'bin']=pd.qcut(z.loc[positive,'x'],bins,duplicates='drop').astype(str)
    out=z.groupby('bin',observed=True).agg(n=('buyer','size'),mean_x=('x','mean'),buyers=('buyer','sum'),rate=('buyer','mean')).reset_index()
    out['empirical_logit']=np.log((out.buyers+.5)/(out.n-out.buyers+.5))
    return out.sort_values('mean_x')
for col in ['Informational_Duration','ProductRelated_Duration','BounceRates']:
    print(f'Marginal binned empirical logit ({col}; one zero group, positive ties preserved):')
    display(marginal_elogit(col).round(4))
print('These marginal plots/tables generate hypotheses only; they do not establish conditional logit linearity or validate Gaussian assumptions.')

In [ ]:
class BinaryMagnitude(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None): return self
    def transform(self,X): return (np.asarray(X,dtype=float)>0).astype(float)

def make_preprocessor(columns, family):
    nums=[x for x in NUMERIC_ALL if x in columns]
    cats=[x for x in CAT if x in columns]
    if family=='BernoulliNB': num_transform=BinaryMagnitude()
    else: num_transform='passthrough'
    drop='first' if family in ['Logit','LDA'] else None
    return ColumnTransformer([('numeric',num_transform,nums),
      ('categorical',OneHotEncoder(handle_unknown='ignore',drop=drop,sparse_output=False),cats),
      ('weekend','passthrough',['Weekend'])],remainder='drop',sparse_threshold=0.0,verbose_feature_names_out=False)

def make_estimator(family):
    if family=='Logit': return LogisticRegression(C=np.inf,solver='newton-cg',tol=1e-8,max_iter=2000)
    if family=='GaussianNB': return GaussianNB()
    if family=='BernoulliNB': return BernoulliNB(binarize=None)
    if family=='LDA': return LinearDiscriminantAnalysis(solver='svd')
    raise ValueError(family)

def make_pipeline(columns,family,estimator=None):
    return Pipeline([('pre',make_preprocessor(columns,family)),('clf',clone(estimator) if estimator is not None else make_estimator(family))])

CONFIGS={
 'Logit':{'family':'Logit','columns':ALL},
 'GaussianNB':{'family':'GaussianNB','columns':ALL},
 'BernoulliNB':{'family':'BernoulliNB','columns':ALL},
 'LDA':{'family':'LDA','columns':ALL}}

def pure_category_levels(X,y,cats):
    result=[]
    for col in cats:
        tab=pd.DataFrame({'level':X[col].astype(str).to_numpy(),'y':np.asarray(y)}).groupby('level').y.agg(['size','sum'])
        pure=tab[(tab['sum']==0)|(tab['sum']==tab['size'])]
        for level,r in pure.iterrows(): result.append(f'{col}={level}:n{int(r["size"])}:y{int(r["sum"])}')
    return result

def design_diagnostics(pre,Xfit):
    Z=np.asarray(pre.transform(Xfit),dtype=float)
    rank=int(np.linalg.matrix_rank(Z)); p=Z.shape[1]
    std=Z.std(axis=0); keep=std>1e-12
    Zn=(Z[:,keep]-Z[:,keep].mean(axis=0))/std[keep] if keep.any() else np.empty((len(Z),0))
    cond=float(np.linalg.cond(np.column_stack([np.ones(len(Z)),Zn]))) if Zn.shape[1] else np.nan
    return Z,rank,p,cond

def positive_probability(est,Z):
    classes=np.asarray(est.classes_); idx=np.flatnonzero(classes==1)
    if len(idx)!=1: raise ValueError(f'Expected one positive class 1, found {classes}')
    return np.asarray(est.predict_proba(Z)[:,idx[0]],dtype=float)

def finite_fitted_parameters(est):
    names=['coef_','intercept_','theta_','var_','class_prior_','class_log_prior_','feature_log_prob_','scalings_','means_','xbar_','priors_']
    checked={name:bool(np.isfinite(np.asarray(getattr(est,name))).all()) for name in names if hasattr(est,name)}
    return bool(checked) and all(checked.values()),checked

def logit_nll(est,Z,y):
    decision=np.asarray(est.decision_function(Z),dtype=float)
    return float(np.sum(np.logaddexp(0.0,decision)-np.asarray(y,dtype=float)*decision))

def unknown_category_counts(Xfit,Xval,cats):
    out={}
    for col in cats:
        known=set(Xfit[col].astype(str)); unseen=~Xval[col].astype(str).isin(known)
        out[col]={'count':int(unseen.sum()),'levels':sorted(Xval.loc[unseen,col].astype(str).unique().tolist())}
    return out

def compact_numerical_label(result):
    value=result.get('numerical_status','UNKNOWN')
    if value.startswith('PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED'):
        return 'Boundary MLE; IRLS reference stable'
    if value=='IRLS_NUMERICAL_REFERENCE_GATE_FAILED':
        return 'IRLS reference gate failed'
    if value=='FINITE_WITH_WARNINGS_REVIEW_REQUIRED':
        return 'Finite fit; warnings logged'
    if value=='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED':
        return 'Finite fit; assumptions apply'
    return value

def run_spec(label,family,columns):
    folds=[]; oof=np.full(len(ytr),np.nan); warning_rows=[]
    cats=[x for x in CAT if x in columns]
    for fold,(fit_idx,val_idx) in enumerate(FOLD_INDICES,1):
        fold_start=time.perf_counter(); mark_progress(label,fold,'fold_start',n_fit=len(fit_idx),n_validation=len(val_idx))
        Xfit=Xtr.iloc[fit_idx][columns]; yfit=ytr.iloc[fit_idx]
        Xval=Xtr.iloc[val_idx][columns]; yval=ytr.iloc[val_idx]
        pipe=make_pipeline(columns,family)
        caught=[]; rank=p=None; cond=np.nan
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                # Preprocessing, estimator fitting and validation transform/prediction are all fold-contained.
                pipe.fit(Xfit,yfit)
                fitted_est=pipe.named_steps['clf']
                mark_progress(label,fold,'fit_complete',fit_seconds=round(time.perf_counter()-fold_start,3),
                    n_iter=int(np.max(fitted_est.n_iter_)) if hasattr(fitted_est,'n_iter_') else None,
                    max_abs_coefficient=float(np.max(np.abs(fitted_est.coef_))) if hasattr(fitted_est,'coef_') else None)
                pre=pipe.named_steps['pre']; est=pipe.named_steps['clf']
                diagnostics_start=time.perf_counter(); mark_progress(label,fold,'design_diagnostics_start')
                Zfit,rank,p,cond=design_diagnostics(pre,Xfit)
                mark_progress(label,fold,'design_diagnostics_complete',diagnostics_seconds=round(time.perf_counter()-diagnostics_start,3),rank=rank,columns=p,condition=cond)
                Zval=np.asarray(pre.transform(Xval),dtype=float)
                prob=positive_probability(est,Zval)
            finite_params,parameter_checks=finite_fitted_parameters(est)
            finite_prob=bool(np.isfinite(prob).all())
            prob_in_range=bool(finite_prob and np.all((prob>=0)&(prob<=1)))
            if not finite_params: raise ValueError(f'non-finite fitted parameters: {parameter_checks}')
            if not prob_in_range: raise ValueError('positive-class probabilities are non-finite or outside [0,1]')
            oof[val_idx]=prob
            pure=pure_category_levels(Xfit,yfit,cats)
            unknown=unknown_category_counts(Xfit,Xval,cats)
            row={'candidate':label,'family':family,'fold':fold,'fit_status':'OK','n_fit':len(fit_idx),'n_validation':len(val_idx),
                 'roc_auc':roc_auc_score(yval,prob),'average_precision':average_precision_score(yval,prob),
                 'design_rank':rank,'design_columns':p,'normalized_condition_number':cond,
                 'finite_model_parameters':finite_params,'parameter_checks':json.dumps(parameter_checks),
                 'finite_probabilities':finite_prob,'probabilities_in_0_1':prob_in_range,
                 'probability_min':float(prob.min()),'probability_max':float(prob.max()),
                 'max_abs_coefficient':float(np.max(np.abs(est.coef_))) if hasattr(est,'coef_') else np.nan,
                 'n_iter':int(np.max(est.n_iter_)) if hasattr(est,'n_iter_') else np.nan,
                 'training_nll':logit_nll(est,Zfit,yfit) if family=='Logit' else np.nan,
                 'pure_nominal_count':len(pure),'pure_nominal_levels':'; '.join(pure),
                 'unseen_validation_levels':json.dumps(unknown)}
            if family=='Logit':
                # Same unpenalised binomial likelihood, independent IRLS numerical reference.
                # Admission is based only on exact training NLL and label-free validation probabilities.
                reference_warn=[]; reference_error=''; ref=None
                Afit=np.column_stack([np.ones(len(Zfit)),Zfit]); Aval=np.column_stack([np.ones(len(Zval)),Zval])
                start=np.r_[float(est.intercept_[0]),np.asarray(est.coef_[0],dtype=float)]
                with warnings.catch_warnings(record=True) as reference_warn:
                    warnings.simplefilter('always')
                    try:
                        ref=sm.GLM(yfit.to_numpy(),Afit,family=sm.families.Binomial()).fit(
                            start_params=start,maxiter=200,tol=1e-10,method='IRLS',wls_method='pinv',disp=0)
                        ref_params=np.asarray(ref.params,dtype=float)
                        ref_eta_fit=Afit@ref_params; ref_eta_val=Aval@ref_params
                        ref_prob=np.asarray(ref.predict(Aval),dtype=float)
                        ref_nll=float(np.mean(np.logaddexp(0.0,ref_eta_fit)-yfit.to_numpy()*ref_eta_fit))
                        primary_mean_nll=float(row['training_nll']/len(yfit))
                        ref_finite=bool(np.isfinite(ref_params).all() and np.isfinite(ref_eta_fit).all() and np.isfinite(ref_eta_val).all()
                            and np.isfinite(ref_prob).all() and np.all((ref_prob>=0)&(ref_prob<=1)))
                        nll_delta=float(abs(ref_nll-primary_mean_nll))
                        p_delta=float(np.max(np.abs(ref_prob-prob)))
                        ref_converged=bool(ref.converged)
                    except Exception as ref_exc:
                        reference_error=f'{type(ref_exc).__name__}: {str(ref_exc)[:250]}'
                        ref_finite=False; nll_delta=np.nan; p_delta=np.nan; ref_converged=False
                for w in reference_warn:
                    warning_rows.append({'candidate':label,'fold':fold,'warning_type':'IRLS '+w.category.__name__,'message':str(w.message)[:300]})
                reference_pass=bool(ref_finite and ref_converged and nll_delta<=1e-6 and p_delta<=1e-3)
                row.update({'reference_solver':'statsmodels GLM Binomial IRLS','reference_wls_method':'pinv',
                    'reference_tol':1e-10,'reference_maxiter':200,
                    'reference_iterations':int(ref.fit_history.get('iteration',-1)) if ref is not None else np.nan,
                    'reference_converged':ref_converged,'reference_finite_parameters_and_probabilities':ref_finite,
                    'reference_mean_training_nll':ref_nll if ref_finite else np.nan,
                    'reference_abs_mean_training_nll_delta':nll_delta,'reference_max_abs_validation_probability_delta':p_delta,
                    'reference_passes_fixed_criteria':reference_pass,'reference_error':reference_error,
                    'reference_warnings':json.dumps([{'type':w.category.__name__,'message':str(w.message)} for w in reference_warn])})
            if family=='Logit':
                if not row.get('reference_passes_fixed_criteria',False):
                    row['numerical_status']='IRLS_NUMERICAL_REFERENCE_GATE_FAILED'
                elif row['pure_nominal_count']>0:
                    row['numerical_status']='PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED_NO_FINITE_MLE_CERTIFICATION'
                else:
                    row['numerical_status']='IRLS_NUMERICAL_REFERENCE_GATE_PASSED'
            else:
                row['numerical_status']='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED'
            for w in caught: warning_rows.append({'candidate':label,'fold':fold,'warning_type':w.category.__name__,'message':str(w.message)[:300]})
            folds.append(row)
            mark_progress(label,fold,'fold_complete',fold_seconds=round(time.perf_counter()-fold_start,3),
                roc_auc=row['roc_auc'],numerical_status=row['numerical_status'])
        except Exception as exc:
            for w in caught: warning_rows.append({'candidate':label,'fold':fold,'warning_type':w.category.__name__,'message':str(w.message)[:300]})
            mark_progress(label,fold,'fold_failed',fold_seconds=round(time.perf_counter()-fold_start,3),failure=f'{type(exc).__name__}: {str(exc)[:250]}')
            folds.append({'candidate':label,'family':family,'fold':fold,'n_fit':len(fit_idx),'n_validation':len(val_idx),
                          'design_rank':rank,'design_columns':p,'normalized_condition_number':cond,
                          'fit_status':'FAILED','failure':f'{type(exc).__name__}: {str(exc)[:250]}'})
    complete=bool(np.isfinite(oof).all()) and len(folds)==K and all(r.get('fit_status')=='OK' for r in folds)
    statuses=[r.get('numerical_status','FAILED') for r in folds]
    if not complete: numerical_status='SCORING_FAILED'
    elif family=='Logit' and 'IRLS_NUMERICAL_REFERENCE_GATE_FAILED' in statuses: numerical_status='IRLS_NUMERICAL_REFERENCE_GATE_FAILED'
    elif family=='Logit' and any('PURE_LEVEL_BOUNDARY_MLE' in x for x in statuses): numerical_status='PURE_LEVEL_BOUNDARY_MLE_IRLS_GATE_PASSED_NO_FINITE_MLE_CERTIFICATION'
    elif family=='Logit': numerical_status='IRLS_NUMERICAL_REFERENCE_GATE_PASSED'
    elif warning_rows: numerical_status='FINITE_WITH_WARNINGS_REVIEW_REQUIRED'
    else: numerical_status='FINITE_PARAMETERS_AND_PROBABILITIES_REVIEW_REQUIRED'
    result={'candidate':label,'family':family,'stage':'all17' if len(columns)==len(ALL) else 'operational16',
            'status':'SCORED' if complete else 'FAILED','numerical_status':numerical_status,'folds':folds,'oof':oof if complete else None,
            'warnings':warning_rows,'failure':'' if complete else 'one or more folds failed or OOF scores are incomplete'}
    if complete:
        auc=np.array([r['roc_auc'] for r in folds]); ap=np.array([r['average_precision'] for r in folds])
        result.update({'mean_fold_auc':auc.mean(),'fold_auc_sd':auc.std(ddof=1),'fold_auc_se':auc.std(ddof=1)/np.sqrt(K),
                       'mean_fold_ap':ap.mean(),'fold_ap_sd':ap.std(ddof=1),'fold_ap_se':ap.std(ddof=1)/np.sqrt(K),
                       'pooled_oof_auc':roc_auc_score(ytr,oof),'pooled_oof_ap':average_precision_score(ytr,oof)})
    return result

RESULTS={}; FOLD_ROWS=[]; WARNING_ROWS=[]
print('Encoders and estimator instances are newly fit inside each training fold; validation rows are transformed only with that fold’s fitted encoder.')

## 5. First pass: four all-17 diagnostic models (A5)

**Purpose.** Follow the “start with all predictors” instruction and document the effect of including `PageValues`. **Method.** Fit the four prescribed unscaled, unregularised/course-likelihood baselines using all 17 source predictors and the common five folds. Every category encoder is learned within its fit fold. **Implication.** `PageValues` association is not proof of leakage; operational models exclude it because availability at voucher decision time is uncertain.

In [ ]:
# A5 — diagnostic first pass, all 17 predictors
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    label=f'All17 | {family}'
    print('Fitting',label,flush=True)
    RESULTS[label]=run_spec(label,family,ALL)
    FOLD_ROWS.extend(RESULTS[label]['folds']); WARNING_ROWS.extend(RESULTS[label]['warnings'])
ALL17=pd.DataFrame([{'candidate':r['candidate'],'scoring status':r['status'],'numerical status':compact_numerical_label(r),'mean ROC-AUC':r.get('mean_fold_auc',np.nan),
 'AUC SD':r.get('fold_auc_sd',np.nan),'AUC SE':r.get('fold_auc_se',np.nan),'mean AP':r.get('mean_fold_ap',np.nan),
 'pooled OOF AUC':r.get('pooled_oof_auc',np.nan),'warnings':len(r['warnings']),'failure':r['failure']} for r in RESULTS.values() if r['stage']=='all17']).set_index('candidate')
display(ALL17.round(4))
fold_diag=pd.DataFrame([x for r in RESULTS.values() if r['stage']=='all17' for x in r['folds']])
all17_audit=fold_diag.groupby('candidate').agg(rank_min=('design_rank','min'),rank_max=('design_rank','max'),
    encoded_columns_min=('design_columns','min'),encoded_columns_max=('design_columns','max'),
    all_parameters_finite=('finite_model_parameters','all'),all_probabilities_in_range=('probabilities_in_0_1','all'))
display(all17_audit)
logit_diag=fold_diag[fold_diag.family.eq('Logit')].copy()
display(logit_diag[['fold','roc_auc','design_rank','design_columns','n_iter','pure_nominal_count','reference_abs_mean_training_nll_delta','reference_max_abs_validation_probability_delta','reference_passes_fixed_criteria']].round(6))
print('Full category support, unseen validation level values/counts, fitted-parameter checks, warning text and diagnostics are saved in artifacts.')
print('All17 diagnostic models remain outside later winner selection.')

**All-17 interpretation.** These are diagnostic comparisons, not evidence that `PageValues` causally informs a purchase. Its timing relative to complete-session information is not verified here, so it is excluded from the operational baseline as a possible downstream measure. The fold table and artifacts retain actual convergence warnings, pure nominal levels, finite parameters and probabilities, and training objective / validation probability comparison to an unpenalised binomial IRLS reference.

In [ ]:
# PageValues descriptive association on training rows only
pv=Xtr.assign(Revenue=ytr).groupby(Xtr.PageValues.gt(0).map({False:'PageValues = 0',True:'PageValues > 0'})).Revenue.agg(sessions='size',buyers='sum',buyer_rate='mean')
display(pv)
print('The above difference is descriptive. PageValues is excluded from operational predictors because its availability before a hypothetical decision is uncertain, not because association proves leakage.')

## 6. Four Operational16 baselines (A6)

**Purpose.** Establish the four course-family baselines on the same 16 source predictors before any optional representation change. **Method.** Use raw numeric magnitudes for unpenalised Logit, GaussianNB and LDA; convert numeric magnitudes to `>0` for BernoulliNB. Encode the six nominal predictors with k−1 indicators for Logit/LDA and full k indicators for both NB models. `Weekend` remains numeric 0/1. No scaling, regularisation, class weighting, automatic transforms or feature engineering are used. **Implication.** Compare common-fold ROC-AUC/AP and numerical validity; the different likelihood inputs are part of the declared course methods.

In [ ]:
# A6 — four operational 16-source-predictor baselines
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    label=f'Operational16 | {family}'
    print('Fitting',label,flush=True)
    RESULTS[label]=run_spec(label,family,SAFE)
    FOLD_ROWS.extend(RESULTS[label]['folds']); WARNING_ROWS.extend(RESULTS[label]['warnings'])
OP16=pd.DataFrame([{'candidate':r['candidate'],'scoring status':r['status'],'numerical status':compact_numerical_label(r),'mean ROC-AUC':r.get('mean_fold_auc',np.nan),
 'AUC SD':r.get('fold_auc_sd',np.nan),'AUC SE':r.get('fold_auc_se',np.nan),'mean AP':r.get('mean_fold_ap',np.nan),
 'pooled OOF AUC':r.get('pooled_oof_auc',np.nan),'warnings':len(r['warnings']),'failure':r['failure']} for r in RESULTS.values() if r['stage']=='operational16']).set_index('candidate')
display(OP16.round(4))
op16_fold_diag=pd.DataFrame([x for r in RESULTS.values() if r['stage']=='operational16' for x in r['folds']])
op16_audit=op16_fold_diag.groupby('candidate').agg(rank_min=('design_rank','min'),rank_max=('design_rank','max'),
    encoded_columns_min=('design_columns','min'),encoded_columns_max=('design_columns','max'),
    all_parameters_finite=('finite_model_parameters','all'),all_probabilities_in_range=('probabilities_in_0_1','all'))
display(op16_audit)
# Direct same-fold AUC changes quantify the effect of omitting PageValues in each family.
comparison=[]
for family in ['Logit','GaussianNB','BernoulliNB','LDA']:
    a=RESULTS[f'All17 | {family}']; b=RESULTS[f'Operational16 | {family}']
    if a['status']=='SCORED' and b['status']=='SCORED':
        for x,y in zip(a['folds'],b['folds']): comparison.append({'family':family,'fold':x['fold'],'AUC all17':x['roc_auc'],'AUC op16':y['roc_auc'],'op16 minus all17':y['roc_auc']-x['roc_auc']})
PAGE_COMPARE=pd.DataFrame(comparison)
PAGE_SUMMARY=PAGE_COMPARE.groupby('family')['op16 minus all17'].agg(['mean','std']).rename(columns={'mean':'mean AUC delta','std':'fold SD'}).round(4)
display(PAGE_SUMMARY)
for family,r in PAGE_SUMMARY.iterrows(): print(f"{family}: excluding PageValues changed mean fold AUC by {r['mean AUC delta']:+.4f} (fold SD of paired change {r['fold SD']:.4f}); this is descriptive, not proof of leakage.")
display(pd.DataFrame(WARNING_ROWS,columns=['candidate','fold','warning_type','message']))
print('Baselines are raw and unscaled. Logit has C=inf (no penalty); GaussianNB sees numeric values and full k one-hot columns; BernoulliNB sees >0 numeric indicators and full k one-hot columns; LDA uses original numerics and k−1 one-hot columns.')
print('Scoring status and numerical review status are separate. The IRLS gate checks same-likelihood training objective and label-free validation probabilities. A passed comparison does not certify a finite MLE when a nominal level is pure in the fit fold.')
for label,r in RESULTS.items(): print(label, 'scoring=',r['status'],'numerical=',r['numerical_status'],'warnings=',len(r['warnings']))

**Operational baseline interpretation and gate.** These four specifications define the reference comparison, not a winner decision. The Logit numerical gate compares the unpenalised cold-start `newton-cg` fit (`tol=1e-8`) with a same-likelihood Binomial IRLS reference using training mean NLL and label-free validation probabilities. Pure nominal levels still mean a finite maximum-likelihood estimate is not certified, even when the numerical comparison passes. The fold table records all warnings, unseen levels, parameter checks, and probabilities.

In [ ]:
# Persist compact actual evidence; no model selection is performed in this stage.
summary_rows=[]
for label,r in RESULTS.items():
    summary_rows.append({'candidate':label,'stage':r['stage'],'scoring_status':r['status'],'numerical_status':r['numerical_status'],
      'mean_fold_auc':r.get('mean_fold_auc',np.nan),'fold_auc_sd':r.get('fold_auc_sd',np.nan),'fold_auc_se':r.get('fold_auc_se',np.nan),
      'mean_fold_ap':r.get('mean_fold_ap',np.nan),'fold_ap_sd':r.get('fold_ap_sd',np.nan),'fold_ap_se':r.get('fold_ap_se',np.nan),
      'pooled_oof_auc':r.get('pooled_oof_auc',np.nan),'pooled_oof_ap':r.get('pooled_oof_ap',np.nan),'warning_count':len(r['warnings']),'failure':r['failure']})
pd.DataFrame(summary_rows).to_csv(ARTIFACTS/'baseline_summary.csv',index=False)
pd.DataFrame(FOLD_ROWS).to_csv(ARTIFACTS/'baseline_fold_metrics_diagnostics.csv',index=False)
pd.DataFrame(WARNING_ROWS,columns=['candidate','fold','warning_type','message']).to_csv(ARTIFACTS/'baseline_warnings.csv',index=False)
for label,r in RESULTS.items():
    if r['oof'] is not None:
        slug=label.replace('All17 | ','all17_').replace('Operational16 | ','operational16_').replace('GaussianNB','gaussian_nb').replace('BernoulliNB','bernoulli_nb').replace(' ','_')
        pd.DataFrame({'row_id':Xtr.index,'observed':ytr.to_numpy(),'oof_probability':r['oof']}).to_csv(ARTIFACTS/(slug+'_oof.csv'),index=False)
print('Saved split/fold provenance, OOF scores, fold metrics/diagnostics, warnings and summary under artifacts/. No revisions or test evaluation are included.')
print(f'Elapsed notebook modelling-stage time: {time.perf_counter()-RUN_START:.1f} seconds.')